In [1]:
%load_ext autoreload
%autoreload 2

from statsmodels.base.model import GenericLikelihoodModel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy.stats as stats
import sys
import re

from scipy.interpolate import make_interp_spline, BSpline
from scipy.stats import multivariate_normal

sys.path.append("../")
import vuong_tests12 as vuong_tests_fast

In [2]:
class JointNormal1(GenericLikelihoodModel):
    
    def loglikeobs(self, params):
        data = np.concatenate([[self.endog],self.exog.transpose()],axis=0)
        mult_rv = stats.multivariate_normal([params[0], 0.0], [[1,0],[0,1]])
        return mult_rv.logpdf(data.transpose())
    
    
class JointNormal2(GenericLikelihoodModel):
    
    def loglikeobs(self, params):
        data = np.concatenate([[self.endog],self.exog.transpose()],axis=0)
        mult_rv = stats.multivariate_normal([0.0, params[0]], [[1,0],[0,1]])
        return mult_rv.logpdf(data.transpose())


def setup_shi(yn,xn):
    # model 1 grad, etc.
    nobs = yn.shape[0]
    model1_param = np.array([yn.mean()])
    model2_param = np.array([xn.mean()])
    
    model1_deriv = JointNormal1(yn,xn)
    ll1 = model1_deriv.loglikeobs(model1_param)
    grad1 =  model1_deriv.score_obs(model1_param).reshape( (nobs,1) )
    hess1 = model1_deriv.hessian(model1_param)
    
    
    model2_deriv = JointNormal2(yn,xn)
    ll2 = model2_deriv.loglikeobs(model2_param)
    grad2 =  model2_deriv.score_obs(model2_param).reshape( (nobs,1) )  
    hess2 = model2_deriv.hessian(model2_param)
    
    return ll1,grad1,hess1,model1_param,ll2,grad2,hess2,model2_param

def gen_data(beta= 1.5, nobs=1000):
    cov = [[25, 0], [0, 1]]
    data = np.random.multivariate_normal([beta,beta], [[25,0],[0,1]],  nobs)
    return data[:,0],data[:,1],nobs

yn,xn,nobs = gen_data()
ll1,grad1,hess1,params1,ll2,grad2,hess2,params2 = setup_shi(yn,xn)
print(grad1.shape,hess1.shape)
#NOTE! Weird size distortions with shi's test when theta = .5....

(1000, 1) (1, 1)


In [3]:
num_sims = 1000
trials =1000
adapt_c = True
data_tuned_epsilon = True
epsilon = .5
biascorrect = True

In [4]:
def get_size_vector(mc_out, data_tuned_epsilon=True):
    # Same return positions as before.
    # boot2 is now unused; boot3 is the tuned SINGLETON bootstrap.
    reg, twostep, sw, boot1, _, sw_test_opt, boot3, shi = mc_out[:8]

    size_vec = [
        1 - reg[0],
        1 - twostep[0],
        1 - sw[0],
        1 - boot1[0],
        1 - shi[0],
    ]

    if data_tuned_epsilon:
        size_vec += [
            1 - sw_test_opt[0],
            1 - boot3[0],
        ]

    return size_vec


def run_null_size_table(
    sample_sizes, num_sims, trials, epsilon,
    data_tuned_epsilon, adapt_c, alpha=.05
):
    headers = [
        "Sample",
        "Normal",
        "Two-Step",
        "SW (fixed)",
        "Singleton (fixed)",
        "Shi (2015)",
    ]

    if data_tuned_epsilon:
        headers += ["SW (tuned)", "Singleton (tuned)"]

    table = []

    for nobs in sample_sizes:
        setup_shi_ex = lambda yn, xn: setup_shi(yn, xn)
        gen_data_ex = lambda: gen_data(nobs=nobs, beta=0)

        mc_out = vuong_tests_fast.monte_carlo(
            num_sims,
            gen_data_ex,
            setup_shi_ex,
            trials=trials,
            epsilon=epsilon,
            data_tuned_epsilon=data_tuned_epsilon,
            adapt_c=adapt_c,
            print_stuff=False,
            alpha=alpha,
            biascorrect=biascorrect
        )

        size_vec = get_size_vector(
            mc_out,
            data_tuned_epsilon=data_tuned_epsilon
        )

        table.append([nobs] + [f"{x:.3f}" for x in size_vec])
        print(f"Finished n={nobs}")

    print(r"\begin{tabular}{|" + "c|" * len(headers) + "}")
    print(r"\hline")
    print(" & ".join(headers) + r" \\ \hline \hline")

    for row in table:
        print(" & ".join(str(y) for y in row) + r" \\")

    print(r"\hline")
    print(r"\end{tabular}")

    return table

# original example from the paper

In [5]:
# Set your globals as needed
sample_sizes = [100, 200, 500,1000]
table = run_null_size_table(
    sample_sizes=sample_sizes,
    num_sims=num_sims,                  # you set this already
    trials=trials,                      # you set this already
    epsilon=0.5,
    data_tuned_epsilon=data_tuned_epsilon,
    adapt_c=adapt_c
)

print(table)

Finished n=100
Finished n=200
Finished n=500
Finished n=1000
\begin{tabular}{|c|c|c|c|c|c|c|c|}
\hline
Sample & Normal & Two-Step & SW (fixed) & Singleton (fixed) & Shi (2015) & SW (tuned) & Singleton (tuned) \\ \hline \hline
100 & 0.000 & 0.000 & 0.044 & 0.042 & 0.000 & 0.041 & 0.042 \\
200 & 0.000 & 0.000 & 0.057 & 0.054 & 0.000 & 0.050 & 0.058 \\
500 & 0.000 & 0.000 & 0.061 & 0.064 & 0.000 & 0.056 & 0.047 \\
1000 & 0.000 & 0.000 & 0.058 & 0.056 & 0.000 & 0.052 & 0.049 \\
\hline
\end{tabular}
[[100, '0.000', '0.000', '0.044', '0.042', '0.000', '0.041', '0.042'], [200, '0.000', '0.000', '0.057', '0.054', '0.000', '0.050', '0.058'], [500, '0.000', '0.000', '0.061', '0.064', '0.000', '0.056', '0.047'], [1000, '0.000', '0.000', '0.058', '0.056', '0.000', '0.052', '0.049']]


In [6]:
# Set your globals as needed
sample_sizes = [100, 200, 500,1000]
table = run_null_size_table(
    sample_sizes=sample_sizes,
    num_sims=num_sims,                  # you set this already
    trials=trials,                      # you set this already
    epsilon=0.5,
    data_tuned_epsilon=data_tuned_epsilon,
    adapt_c=adapt_c
)

print(table)

Finished n=100
Finished n=200
Finished n=500
Finished n=1000
\begin{tabular}{|c|c|c|c|c|c|c|c|}
\hline
Sample & Normal & Two-Step & SW (fixed) & Singleton (fixed) & Shi (2015) & SW (tuned) & Singleton (tuned) \\ \hline \hline
100 & 0.000 & 0.000 & 0.052 & 0.043 & 0.000 & 0.051 & 0.045 \\
200 & 0.000 & 0.000 & 0.037 & 0.038 & 0.000 & 0.035 & 0.036 \\
500 & 0.000 & 0.000 & 0.053 & 0.055 & 0.000 & 0.047 & 0.050 \\
1000 & 0.000 & 0.000 & 0.052 & 0.048 & 0.000 & 0.040 & 0.052 \\
\hline
\end{tabular}
[[100, '0.000', '0.000', '0.052', '0.043', '0.000', '0.051', '0.045'], [200, '0.000', '0.000', '0.037', '0.038', '0.000', '0.035', '0.036'], [500, '0.000', '0.000', '0.053', '0.055', '0.000', '0.047', '0.050'], [1000, '0.000', '0.000', '0.052', '0.048', '0.000', '0.040', '0.052']]


In [ ]:
# Set your globals as needed
sample_sizes = [100, 200, 500,1000]
table = run_null_size_table(
    sample_sizes=sample_sizes,
    num_sims=num_sims,                  # you set this already
    trials=trials,                      # you set this already
    epsilon=0.5,
    data_tuned_epsilon=data_tuned_epsilon,
    adapt_c=adapt_c
)

print(table)

Finished n=100
Finished n=200
Finished n=500


In [8]:
# Set your globals as needed
table = run_null_size_table(
    sample_sizes=sample_sizes,
    num_sims=num_sims,                  # you set this already
    trials=trials,                      # you set this already
    epsilon=0.5,
    data_tuned_epsilon=data_tuned_epsilon,
    adapt_c=adapt_c,alpha=.025
)

print(table)

Finished n=200
Finished n=500
Finished n=1000
\begin{tabular}{|c|c|c|c|c|c|c|c|}
\hline
Sample & Normal & Two-Step & SW (fixed) & Singleton (fixed) & Shi (2015) & SW (tuned) & Singleton (tuned) \\ \hline \hline
100 & 0.000 & 0.000 & 0.020 & 0.033 & 0.000 & 0.017 & 0.030 \\
200 & 0.000 & 0.000 & 0.025 & 0.029 & 0.000 & 0.022 & 0.030 \\
500 & 0.000 & 0.000 & 0.027 & 0.027 & 0.000 & 0.024 & 0.024 \\
1000 & 0.000 & 0.000 & 0.021 & 0.023 & 0.000 & 0.017 & 0.019 \\
\hline
\end{tabular}
[[100, '0.000', '0.000', '0.020', '0.033', '0.000', '0.017', '0.030'], [200, '0.000', '0.000', '0.025', '0.029', '0.000', '0.022', '0.030'], [500, '0.000', '0.000', '0.027', '0.027', '0.000', '0.024', '0.024'], [1000, '0.000', '0.000', '0.021', '0.023', '0.000', '0.017', '0.019']]


In [9]:
# Set your globals as needed
table = run_null_size_table(
    sample_sizes=sample_sizes,
    num_sims=num_sims,                  # you set this already
    trials=trials,                      # you set this already
    epsilon=0.5,
    data_tuned_epsilon=data_tuned_epsilon,
    adapt_c=adapt_c,alpha=.01
)

print(table)

Finished n=100
Finished n=200
Finished n=500
Finished n=1000
\begin{tabular}{|c|c|c|c|c|c|c|c|}
\hline
Sample & Normal & Two-Step & SW (fixed) & Singleton (fixed) & Shi (2015) & SW (tuned) & Singleton (tuned) \\ \hline \hline
100 & 0.000 & 0.000 & 0.005 & 0.006 & 0.000 & 0.004 & 0.005 \\
200 & 0.000 & 0.000 & 0.008 & 0.010 & 0.000 & 0.007 & 0.011 \\
500 & 0.000 & 0.000 & 0.014 & 0.016 & 0.000 & 0.012 & 0.012 \\
1000 & 0.000 & 0.000 & 0.009 & 0.010 & 0.000 & 0.007 & 0.008 \\
\hline
\end{tabular}
[[100, '0.000', '0.000', '0.005', '0.006', '0.000', '0.004', '0.005'], [200, '0.000', '0.000', '0.008', '0.010', '0.000', '0.007', '0.011'], [500, '0.000', '0.000', '0.014', '0.016', '0.000', '0.012', '0.012'], [1000, '0.000', '0.000', '0.009', '0.010', '0.000', '0.007', '0.008']]
